# Mineria de datos clase 8
## Alumno: Enzo Ariel Melian

Contexto:
- Trabajas en una empresa de seguros y necesitas desarrollar modelos para: Predecir si un cliente presentará un reclamo (Sí/No) → Regresión Logística.
- Estimar el monto del reclamo en dólares → Regresión Múltiple.

Tareas a realizar
1. Preprocesamiento: Convertir "Reclamo" a numérico (0=No, 1=Sí). Dividir en entrenamiento (80%) y prueba (20%).
2. Regresión Logística: Variables predictoras: Edad, Ingresos, Historial de Accidentes. Evaluar con Matriz de Confusión y Precisión.
3. Regresión Múltiple: Mismas variables predictoras. Evaluar con R², MSE y MAE.
4. Interacción con IA (LLM): Consulta sugerida: "Tengo un dataset de aseguradoras. ¿Qué estrategias de modelado recomendarías para predecir el monto de un reclamo?"

## Carga del dataset

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
     accuracy_score, confusion_matrix, classification_report, r2_score,
     mean_squared_error, mean_absolute_error
     )

data = { "Edad": [25, 40, 32, 50, 28, 45],
         "Ingresos": [30000, 55000, 42000, 60000, 37000, 52000],
          "Historial_Accidentes": [1, 0, 1, 0, 1, 0],
          "Monto_Reclamo": [5000, 0, 7500, 0, 6500, 0],
          "Reclamo": [1, 0, 1, 0, 1, 0]
        }

df = pd.DataFrame(data)
df

,Edad,Ingresos,Historial_Accidentes,Monto_Reclamo,Reclamo
0,25,30000,1,5000,1
1,40,55000,0,0,0
2,32,42000,1,7500,1
3,50,60000,0,0,0
4,28,37000,1,6500,1
5,45,52000,0,0,0


## 1. Preprocesamiento

In [5]:
X = df[["Edad", "Ingresos", "Historial_Accidentes"]]
y_c = df["Reclamo"]
y_r = df["Monto_Reclamo"]

### 1.1 Entrenamiento

In [7]:
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X, y_c, test_size=0.2, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y_r, test_size=0.2, random_state=42)

## 2. Regresión logística

In [8]:
log_model = LogisticRegression(max_iter=1000).fit(X_tr_c, y_tr_c)
y_pred_c = log_model.predict(X_te_c)
print("Precisión:", accuracy_score(y_te_c, y_pred_c))
print("Matriz de Confusión: ", confusion_matrix(y_te_c, y_pred_c))

Precisión: 1.0
Matriz de Confusión:  [[1 0]
 [0 1]]


Estos resultados de la regresión logistica nos dicen que:
- Se predijo con una exactitud del 100% las veces que un cliente presentará un reclamo.

## 3. Regresión múltiple

In [9]:
reg_model = LinearRegression().fit(X_tr_r, y_tr_r)
y_pred_r = reg_model.predict(X_te_r)
print(f"R²: {r2_score(y_te_r, y_pred_r):.2f}")
print(f"MSE: {mean_squared_error(y_te_r, y_pred_r):.2f}")
print(f"MAE: {mean_absolute_error(y_te_r, y_pred_r):.2f}")

R²: -4.69
MSE: 35584183.67
MAE: 5464.29


Estos resultados de la regresión múltiple nos dicen que:
- No se pudo predecir el monto del reclamo, si R2 tuviera el valor de 1 (o cercano al mismo) la precisión de la regresión sería alta. En este caso al ser -4.69, nos indica que la predicción no pudo siquiera realizarse.

## 4. Interacción con IA

Le pregunte a Gemini:

1- Tengo un dataset de aseguradoras. ¿Qué estrategias de modelado recomendarías para predecir el monto de un reclamo?

**Su respuesta resumida es:**

*Modelos de Regresión Tradicionales con Transformaciones:*
  - Regresión Lineal Múltiple

*Modelos para Datos con Exceso de Ceros (Zero-Inflated Models):*
  - Modelos de Dos Partes (Two-Part Models o Hurdle Models)

*Modelos de Regresión Generalizados (GLM - Generalized Linear Models):*
  - Regresión Gamma o Inversa Gaussian

*Modelos Basados en Árboles (Tree-Based Models)*
  - Random Forest Regressor
  - Gradient Boosting Regressor (ej., XGBoost, LightGBM, CatBoost)

*Modelos Híbridos o Ensembles:*
  - Combinar la fuerza de varios modelos (por ejemplo, un modelo de dos partes donde la segunda parte es un Random Forest). Stacking o Bagging también pueden mejorar la robustez y la precisión.
  
---------
2- Para la regresión multiple realizada. ¿Que razón habría en este caso el por que no tiene una "precisión positiva"?

**Su respuesta:**

La razón principal por la que la regresión múltiple no obtuvo una "precisión positiva" (refiriéndonos al R² negativo) en este caso específico es la combinación de:

*- Tamaño del Dataset Muy Pequeño: Con solo 6 filas de datos y dividiendo para entrenamiento y prueba, el modelo tiene muy pocos ejemplos para aprender patrones robustos. Para la regresión lineal, en particular, se necesitan más datos para establecer relaciones significativas.*

*- La Presencia de Ceros en la Variable Objetivo (Monto_Reclamo): Como mencioné, Monto_Reclamo tiene muchos valores de cero (3 de 6). Un modelo de regresión lineal estándar asume una distribución continua y aproximadamente normal de los residuos y no está diseñado para manejar eficientemente una variable objetivo con una alta concentración de ceros. Esto introduce sesgos y dificulta que el modelo encuentre una relación lineal coherente.*

*- Posible Falta de Relación Lineal Simple: Las variables predictoras (Edad, Ingresos, Historial de Accidentes) pueden no tener una relación lineal directa con el monto del reclamo, o hay otros factores no incluidos en el dataset que son más influyentes. Un R² negativo sugiere que, para estos datos, incluso un modelo muy simple que predijera siempre el promedio del monto de reclamo habría funcionado mejor que la regresión lineal que entrenamos.
En resumen, la combinación de un dataset muy limitado y la naturaleza de la variable Monto_Reclamo (con muchos ceros) son las razones clave por las que la regresión lineal no pudo ofrecer una "precisión positiva" o un R² aceptable.*
